In [2]:
import gymnasium as gym
import ale_py

def make_env(env_name):
    # Đăng ký các môi trường Atari
    gym.register_envs(ale_py)
    
    # Loại bỏ hoàn toàn render_mode để chạy ở chế độ background (không mở cửa sổ)
    env = gym.make(env_name, obs_type="ram")
    
    # Đã bỏ dòng FrameStackObservation để obs trả về đúng 1 mảng 128 số
    return env

if __name__ == "__main__":
    # Khởi tạo môi trường không có render_mode
    env = make_env("ALE/Pacman-v5")
    
    # obs lúc này là một mảng NumPy phẳng có hình dạng (128,)
    obs, info = env.reset()
    
    print("--- MẢNG RAM BAN ĐẦU (128 số) ---")
    print(obs)
    print(f"Kích thước mảng: {obs.shape}\n")
    
    # Chạy thử 5 bước để xem RAM thay đổi và in ra
    for step in range(5):
        action = env.action_space.sample() 
        obs, reward, terminated, truncated, info = env.step(action)
        
        print(f"--- RAM Ở BƯỚC THỨ {step + 1} ---")
        # Chuyển thành list để in trên một dòng cho bạn dễ nhìn
        print(list(obs)) 
        
        if terminated or truncated:
            obs, info = env.reset()
            
    env.close()

--- MẢNG RAM BAN ĐẦU (128 số) ---
[ 12   0   0   0   6   0   0  19 255 255 254 255 255 255 127 255 255 255
 255 255 255 255 255 255   3   9  60  10   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   9  76  76  76  76  76  76
  66  33  33  33  33 128   0   0   0   0  11   8   9   8   9   8   8   8
   8   6   0 255   0   0   0   0   0   0  64 160   0   0   0   0 254  78
 254   7   6 142 253 178 253 255   0   0  59   0   0 136   0   0   0   0
  12   0  79 255   7   8 111  47   9  87  15 251   4 251 248  53 105  52
 211 247]
Kích thước mảng: (128,)

--- RAM Ở BƯỚC THỨ 1 ---
[np.uint8(16), np.uint8(0), np.uint8(0), np.uint8(0), np.uint8(6), np.uint8(0), np.uint8(0), np.uint8(19), np.uint8(255), np.uint8(255), np.uint8(254), np.uint8(255), np.uint8(255), np.uint8(255), np.uint8(127), np.uint8(255), np.uint8(255), np.uint8(255), np.uint8(255), np.uint8(255), np.uint8(255), np.uint8(255), np.uint8(255), np.uint8(255), np.uint8(3), np.uint8(9), np.uint8(60), np.uint8

In [3]:
import gymnasium as gym
import ale_py
import numpy as np

class PacmanHRAPreprocessing(gym.ObservationWrapper):
    def __init__(self, env):
        super().__init__(env)
        # Định nghĩa lại không gian quan sát (Observation Space) theo logic của bạn
        self.observation_space = gym.spaces.Dict({
            "binary_channels": gym.spaces.Box(low=0, high=1, shape=(11, 40, 40), dtype=np.uint8),
            "direction_vector": gym.spaces.Box(low=0, high=1, shape=(4,), dtype=np.uint8)
        })

    def observation(self, obs):
        # obs gốc từ ALE/Pacman-v5 có kích thước (210, 160, 3)
        
        # 1. Cắt xén hình ảnh (Cropping) thành 160x160
        # Cắt bỏ 30 pixel trên (điểm số) và 20 pixel dưới (thông tin mạng)
        cropped_img = obs[30:190, :, :] 
        
        # 2. Trích xuất tọa độ và tạo 11 kênh nhị phân (40x40)
        # Trong thực tế, bạn cần thuật toán dò màu hoặc đọc RAM song song để tìm tọa độ chính xác.
        # Dưới đây là mô phỏng việc giảm chiều rộng/cao đi 4 lần (160 / 4 = 40)
        binary_channels = np.zeros((11, 40, 40), dtype=np.uint8)
        
        # [Bước xử lý nội bộ: Điền 1 vào các vị trí của Pacman, 4 con ma, hạt pellet...]
        # Ví dụ giả lập:
        binary_channels[0, 20, 20] = 1 # Kênh 0: Ms. Pac-Man đang ở giữa bản đồ (40x40)
        
        # 3. Mã hóa hướng di chuyển (One-hot vector 4 phần tử: Bắc, Đông, Nam, Tây)
        # Giả sử bạn lấy được hướng hiện tại từ game (Ví dụ: đang đi bên ĐÔNG)
        direction_vector = np.array([0, 1, 0, 0], dtype=np.uint8) 
        
        return {
            "binary_channels": binary_channels,
            "direction_vector": direction_vector
        }

def make_env(env_name):
    gym.register_envs(ale_py)
    # BẮT BUỘC bỏ obs_type="ram", sử dụng RGB mặc định để có hình ảnh (210x160)
    env = gym.make(env_name) 
    # Áp dụng bộ tiền xử lý tùy chỉnh vừa viết ở trên
    env = PacmanHRAPreprocessing(env)
    return env

In [4]:
# --- KHỞI TẠO KHÔNG GIAN BẢNG HRA (TABULAR SETUP) ---

# Theo tính toán của bạn: 4 maps × 400 positions × 950 states × 9 actions
NUM_MAPS = 4
NUM_POSITIONS = 400
NUM_STATES = 950
NUM_ACTIONS = 9 # Thường Atari có 9 hoặc 18 hành động

# Khởi tạo bảng Q-Table bằng 0
# Kích thước bộ nhớ xấp xỉ: 14,000,000 ô float32 * 4 bytes = ~56 MB RAM (Rất nhẹ)
hra_q_table = np.zeros((NUM_MAPS, NUM_POSITIONS, NUM_STATES, NUM_ACTIONS), dtype=np.float32)

print(f"Khởi tạo bảng HRA thành công với hình dạng: {hra_q_table.shape}")
print(f"Tổng số mục nhập (entries): {hra_q_table.size:,}")

Khởi tạo bảng HRA thành công với hình dạng: (4, 400, 950, 9)
Tổng số mục nhập (entries): 13,680,000


In [5]:
if __name__ == "__main__":
    # Khởi tạo môi trường Pacman phiên bản v5 mới nhất
    env = make_env("ALE/Pacman-v5")
    obs, info = env.reset()
    
    # In thử cấu trúc dữ liệu sau khi qua bộ lọc xử lý
    print("\n--- CẤU TRÚC ĐẦU VÀO SAU KHI TIỀN XỬ LÝ ---")
    print("Kích thước 11 kênh nhị phân:", obs["binary_channels"].shape)
    print("Vector hướng di chuyển:", obs["direction_vector"])
    
    # Chạy thử 1 bước hành động
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    
    # Thuật toán HRA của bạn sẽ lấy obs này, giải mã thành (Map_ID, Position_ID, State_ID)
    # Sau đó tra cứu trực tiếp vào bảng `hra_q_table[map_id, pos_id, state_id]` để chọn hành động tối ưu.

    env.close()


--- CẤU TRÚC ĐẦU VÀO SAU KHI TIỀN XỬ LÝ ---
Kích thước 11 kênh nhị phân: (11, 40, 40)
Vector hướng di chuyển: [0 1 0 0]


In [1]:
import gymnasium as gym
import ale_py
import numpy as np

def get_hra_identifiers(ram_bytes):
    """
    Hàm giải mã 128 bytes RAM để trả về chính xác Map_ID, Position_ID, State_ID
    (Các chỉ số index dưới đây là ví dụ mô phỏng dựa trên cấu trúc bộ nhớ Atari Pacman)
    """
    # 1. Xác định Map_ID (Pacman có 4 bản đồ đổi theo màn chơi)
    current_level = ram_bytes[11]  # Giả sử ô nhớ 11 lưu level
    map_id = current_level % 4    # Trả về từ 0 đến 3
    
    # 2. Xác định vị trí của Pacman (X, Y)
    pacman_x = ram_bytes[10] # Ô nhớ 10 lưu tọa độ X
    pacman_y = ram_bytes[16] # Ô nhớ 16 lưu tọa độ Y
    
    # Gom nhóm (Map) tọa độ pixel thực tế thành một trong 400 vị trí hợp lệ trên mê cung
    # Ví dụ: Mê cung thực tế là một lưới ô cờ 20x20 đường đi = 400 vị trí
    grid_x = int(pacman_x / 8) # Chia tỉ lệ pixel sang ô lưới
    grid_y = int(pacman_y / 8)
    position_id = (grid_y * 20 + grid_x) % 400 # Trả về từ 0 đến 399
    
    # 3. Xác định Trạng thái (State_ID - hướng đi + vị trí các con ma xung quanh)
    # Kết hợp hướng quay mặt của Pacman (0,1,2,3) và tình trạng nguy hiểm để ra 1 trong 950 trạng thái
    pacman_dir = ram_bytes[2] % 4  # Ô nhớ 2 lưu hướng đi
    ghost_distance = ram_bytes[5]  # Giả sử ô nhớ 5 lưu khoảng cách tới con ma gần nhất
    
    state_id = (pacman_dir * 200 + ghost_distance) % 950 # Trả về từ 0 đến 949
    
    return map_id, position_id, state_id

# --- CHẠY THỬ NGHIỆM ---
if __name__ == "__main__":
    gym.register_envs(ale_py)
    # Khởi tạo với obs_type="ram" để lấy 128 con số bộ nhớ
    env = gym.make("ALE/Pacman-v5", obs_type="ram")
    
    ram_obs, info = env.reset()
    
    # Thử chạy 10 bước và in ra Vị trí, Trạng thái đã được giải mã
    for step in range(5):
        action = env.action_space.sample()
        ram_obs, reward, terminated, truncated, info = env.step(action)
        
        # Trích xuất các biến định danh phục vụ cho bảng HRA Q-Table
        map_id, position_id, state_id = get_hra_identifiers(ram_obs)
        
        print(f"Bước {step+1} -> Map: {map_id} | Vị trí ID: {position_id} | Trạng thái ID: {state_id}")
        
    env.close()

A.L.E: Arcade Learning Environment (version 0.12.0+0706845)
[Powered by Stella]


OverflowError: Python integer 950 out of bounds for uint8